In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [3]:
from waymo_agent.notebook_imports import *

In [4]:
from functools import cache

In [5]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *
from waymo_agent.action_heuristic.heuristic_simple import PricingAgent, DispatchAgent, RepositionAgent
from waymo_agent.models.ppo_model import RideShareActorCritic
from waymo_agent.graph_env.ENV import RideShareEnv
from waymo_agent.models.ppo_model import *

In [6]:
from kret_sandbox.VIS import dtt
from kret_sandbox.exp_decay import exp_decay_half_life, get_gamma_from_half_life

In [7]:
def get_obs_tuple(env: RideShareEnv):
    veh = env.observation_curr["vehicles"]
    req = env.observation_curr["pending_requests"]
    rides = env.observation_curr["active_rides"]
    return veh, req, rides


def get_obs_tuple_from_obs(obs: ObservationDict):
    veh = obs["vehicles"]
    req = obs["pending_requests"]
    rides = obs["active_rides"]
    return veh, req, rides

# Run Sim

In [8]:
config = EnvConfig(max_episode_steps=60 * 3)
plt_cfg = PlotConfig()
env_model = RideShareEnv(config, plt_cfg)

Assigned lambda values to nodes. Total lambda: 2.3211 (target: 2.3460)


In [9]:
GAMMA = get_gamma_from_half_life(config.max_episode_steps // 2)
round(GAMMA, 5)

0.99233

In [ ]:
def discounted_rewards(rewards: np.ndarray, gamma: float = 0.997) -> np.ndarray:
    """
    Discount rewards using discount factor gamma.
    """
    disc_schedule = exp_decay_half_life(len(rewards), gamma=gamma)
    return rewards * disc_schedule


def get_act_dict(agents: tuple[PricingAgent, DispatchAgent, RepositionAgent], obs: ObservationDict) -> ActionDict:
    price_agent, dispatch_agent, reposition_agent = agents
    prices = price_agent.price(obs)
    dispatch_actions = dispatch_agent.dispatch(obs)
    reposition_actions = reposition_agent.reposition(obs)

    action_agent: ActionDict = {
        "prices": prices,
        "dispatch": dispatch_actions,
        "reposition": reposition_actions,
    }
    return action_agent

In [32]:
from multiprocessing.pool import TERMINATE


@torch.no_grad()
def run_model_simulation(
    env_curr: RideShareEnv,
    model: RideShareActorCritic,
    num_iter: int = 1,
    gamma: float = GAMMA,
    deterministic: bool = False,
):
    """
    Returns:
      REWARDS: list[episode][t] discounted reward_t
      OBS:     list[episode][t] obs dict (numpy)
      ACT:     list[episode][t] action dict (numpy)
    """
    model.eval()

    REWARDS: list[np.ndarray] = []
    OBS: list[list[dict[str, np.ndarray]]] = []
    ACT: list[list[dict[str, np.ndarray]]] = []
    TERMINATED: list[bool] = []
    TRUNCATED: list[bool] = []

    for i in tqdm(range(num_iter), desc="model rollout"):
        # tqdm.write(f"Starting episode {i+1}/{num_iter}...")
        obs_np, _info = env_curr.reset()
        terminated = False
        truncated = False
        done = False

        rews_raw: list[float] = []
        obs_list: list[dict[str, np.ndarray]] = []
        act_list: list[dict[str, np.ndarray]] = []

        while not done:
            # tqdm.write(f"Step {env_curr.current_step}/{env_curr.config.max_episode_steps}")
            obs_list.append(obs_np)

            obs_t = obs_pd_to_torch(obs_np)

            act_t = model.act(obs_t, deterministic=deterministic)
            act_np = action_torch_to_numpy(act_t)
            env_curr._validate_action(act_np)
            act_list.append(act_np)

            obs_np, reward, terminated, truncated, _info = env_curr.step(act_np)  # type: ignore[arg-type]
            # print(f"Reward: {reward}")
            rews_raw.append(float(reward))
            done = bool(terminated) or bool(truncated)

        rews = discounted_rewards(np.array(rews_raw, dtype=np.float32), gamma=gamma)
        REWARDS.append(rews)
        OBS.append(obs_list)
        ACT.append(act_list)
        TERMINATED.append(terminated)
        TRUNCATED.append(truncated)

    return REWARDS, OBS, ACT, TERMINATED, TRUNCATED

In [27]:
cfg_ppo = PPOTrainConfig()
model = RideShareActorCritic(env_model)

In [33]:
REWARDS, OBS, ACT, TERMINATED, TRUNCATED = run_model_simulation(env_model, model, num_iter=1)

model rollout:   0%|          | 0/1 [00:00<?, ?it/s]

In [34]:
REWARDS

[array([ 0.        ,  0.        ,  0.29541445, -0.01956232, -0.47602123,
        -1.881373  , -2.3603625 , -3.1610508 , -1.7887241 , -1.4529052 ,
        -2.275876  , -2.351077  , -1.5149163 , -1.8898777 , -2.534958  ,
        -2.788454  , -1.5777261 , -2.108585  , -1.9376845 , -1.8198584 ,
        -0.42927834, -0.6295579 , -2.8216481 , -0.9691362 , -2.56209   ,
        -3.6236994 , -1.9101871 , -1.1151246 , -2.0979183 , -3.693848  ,
        -2.4684508 , -3.1197135 , -2.7728245 , -5.2013483 , -2.3977964 ,
        -4.227892  , -3.744755  , -3.4719148 , -4.436968  , -3.6234028 ,
        -4.1281    , -1.8464171 , -5.9892774 , -3.4255686 , -7.3167343 ,
        -2.6848829 , -5.1926217 , -4.7989364 , -3.5958285 , -4.33262   ,
        -2.623317  , -2.0966437 , -4.0277567 , -3.8042328 , -4.705905  ,
        -4.9288974 , -3.123234  , -3.2959976 , -4.1144624 , -3.6955836 ,
        -2.4481854 , -4.0114164 , -3.419681  , -3.927272  , -2.8293023 ,
        -1.9109459 , -3.0099266 , -4.080157  , -1.8

In [30]:
TERMINATED, TRUNCATED

([False], [True])

In [25]:
env_model._rewards

{'penalty_multiple_dispatch_assignment': 1.0,
 'penalty_assign_to_unavailable_vehicle': -2.0,
 'distance_penalty_xy_normed': 0.0,
 'penalty_rejected': np.float64(-0.0),
 'penalty_expire': np.float64(-0.0),
 'ride_reward_total': np.float64(-6.824303562743239)}

In [14]:
# import importlib
# import waymo_agent.data_classes.enriched_df_base

# _ = importlib.reload(waymo_agent.data_classes.enriched_df_base)

## NN Actor-Critic

In [15]:
_ = env_model.reset()

In [18]:
model, logs = ppo_model.train_ppo(env_model, model, cfg=cfg_ppo, save_path=MODEL_WEIGHT_DIR / "ppo_model_final.pt")

PPO steps:   0%|          | 0/2000 [00:00<?, ?step/s]

PPO update epochs:   0%|          | 0/4 [00:00<?, ?it/s]

PPO update epochs:   0%|          | 0/4 [00:00<?, ?it/s]

PPO update epochs:   0%|          | 0/4 [00:00<?, ?it/s]

PPO update epochs:   0%|          | 0/4 [00:00<?, ?it/s]

...

PPO update epochs:   0%|          | 0/4 [00:00<?, ?it/s]

PPO update epochs:   0%|          | 0/4 [00:00<?, ?it/s]

PPO update epochs:   0%|          | 0/4 [00:00<?, ?it/s]

PPO update epochs:   0%|          | 0/4 [00:00<?, ?it/s]

PPO update epochs:   0%|          | 0/4 [00:00<?, ?it/s]

In [17]:
save_weights(model, MODEL_WEIGHT_DIR / "ppo_model_final.pt")